In [9]:
import pandas as pd
from pathlib import Path

In [ ]:
project_route = Path.cwd().parent
csv_route = project_route / "data" / "raw" / "E0.csv"
df = pd.read_csv(csv_route, encoding="latin1")
df.columns = df.columns.astype(str).str.replace("ï»¿", "", regex=False).str.strip()

## Data cleaning strategy

This notebook organizes data cleaning for extract.py. Two tables:

- **Clean historical match table**: all match information, one row per match
- **Model-safe pre-match table**: only values known before kickoff (no data leakage)

In [13]:
print(list(df.columns))


['Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 'BFH', 'BFD', 'BFA', 'PSH', 'PSD', 'PSA', 'WHH', 'WHD', 'WHA', '1XBH', '1XBD', '1XBA', 'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA', 'BFEH', 'BFED', 'BFEA', 'B365>2.5', 'B365<2.5', 'P>2.5', 'P<2.5', 'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5', 'BFE>2.5', 'BFE<2.5', 'AHh', 'B365AHH', 'B365AHA', 'PAHH', 'PAHA', 'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'BFEAHH', 'BFEAHA', 'B365CH', 'B365CD', 'B365CA', 'BWCH', 'BWCD', 'BWCA', 'BFCH', 'BFCD', 'BFCA', 'PSCH', 'PSCD', 'PSCA', 'WHCH', 'WHCD', 'WHCA', '1XBCH', '1XBCD', '1XBCA', 'MaxCH', 'MaxCD', 'MaxCA', 'AvgCH', 'AvgCD', 'AvgCA', 'BFECH', 'BFECD', 'BFECA', 'B365C>2.5', 'B365C<2.5', 'PC>2.5', 'PC<2.5', 'MaxC>2.5', 'MaxC<2.5', 'AvgC>2.5', 'AvgC<2.5', 'BFEC>2.5', 'BFEC<2.5', 'AHCh', 'B365CAHH', 'B365CAHA', 'PCAHH', 'PCAHA',

In [ ]:
all_columns = {
    'div': 'league_division',
    'date': 'date',
    'time': 'kickoff_time',
    'hometeam': 'home_team',
    'awayteam': 'away_team',
    'referee': 'referee',
    'attendance': 'attendance',
    'ftr': 'result',
    'fthg': 'home_goals',
    'ftag': 'away_goals',
    'hthg': 'half_time_home_goals',
    'htag': 'half_time_away_goals',
    'htr': 'half_time_result',
    'hs': 'home_shots',
    'as': 'away_shots',
    'hst': 'home_shots_on_target',
    'ast': 'away_shots_on_target',
    'hhw': 'home_hit_woodwork',
    'ahw': 'away_hit_woodwork',
    'hc': 'home_corners',
    'ac': 'away_corners',
    'hf': 'home_fouls_committed',
    'af': 'away_fouls_committed',
    'hfkc': 'home_free_kicks_conceded',
    'afkc': 'away_free_kicks_conceded',
    'ho': 'home_offsides',
    'ao': 'away_offsides',
    'hy': 'home_yellow',
    'ay': 'away_yellow',
    'hr': 'home_red',
    'ar': 'away_red',
    'b365h': 'b365_home_win_odds',
    'b365d': 'b365_draw_odds',
    'b365a': 'b365_away_win_odds',
    'avgh': 'market_avg_home_win_odds',
    'avgd': 'market_avg_draw_odds',
    'avga': 'market_avg_away_win_odds',
}

lower_case_columns = {column.lower(): column for column in df.columns}
selected_columns = [lower_case_columns[column] for column in all_columns if column in lower_case_columns]
df = df[selected_columns]
df.rename(columns = {
    lower_case_columns[column]: all_columns[column]
    for column in all_columns if column in lower_case_columns
}, inplace=True)

df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
df['kickoff_time'] = df['kickoff_time'].astype('string').str.strip()
df['match_datetime'] = pd.to_datetime(df['date'].dt.strftime('%Y-%m-%d') + ' ' + df['kickoff_time'], errors = 'coerce')
df.drop(columns = ['date', 'kickoff_time'], errors = 'ignore', inplace=True)

text_columns = [
    'league_division',
    'home_team',
    'away_team',
    'referee',
    'result',
    'half_time_result'
]
for column in text_columns:
    if column in df.columns:
        df[column] = df[column].astype('string').str.strip()

numeric_columns = [
    'attendance',
    'home_goals',
    'away_goals',
    'half_time_home_goals',
    'half_time_away_goals',
    'b365_home_win_odds',
    'b365_draw_odds',
    'b365_away_win_odds',
    'market_avg_home_win_odds',
    'market_avg_draw_odds',
    'market_avg_away_win_odds',
    'home_shots',
    'away_shots',
    'home_shots_on_target',
    'away_shots_on_target',
    'home_hit_woodwork',
    'away_hit_woodwork',
    'home_corners',
    'away_corners',
    'home_fouls_committed',
    'away_fouls_committed',
    'home_free_kicks_conceded',
    'away_free_kicks_conceded',
    'home_offsides',
    'away_offsides',
    'home_yellow',
    'away_yellow',
    'home_red',
    'away_red',
]
for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors='coerce')

df = (df.dropna(subset=['match_datetime', 'home_team', 'away_team']).sort_values(
    ['match_datetime', 'home_team', 'away_team']
    ).reset_index(drop=True)
)

In [ ]:
pre_match_columns = [
    'league_division',
    'date',
    'kickoff_time',
    'match_datetime',
    'home_team',
    'away_team',
    'referee',
    'attendance',
    'b365_home_win_odds',
    'b365_draw_odds',
    'b365_away_win_odds',
    'market_avg_home_win_odds',
    'market_avg_draw_odds',
    'market_avg_away_win_odds',
]

df_model_prematch = df[[column for column in pre_match_columns if column in df.columns]].copy()

df_market_odds = df[[
    column for column in [
        'date',
        'home_team',
        'away_team',
        'b365_home_win_odds',
        'b365_draw_odds',
        'b365_away_win_odds',
        'market_avg_home_win_odds',
        'market_avg_draw_odds',
        'market_avg_away_win_odds',
    ] if column in df.columns
]].copy()

In [38]:
df.head()

,league_division,home_team,away_team,referee,result,home_goals,away_goals,half_time_home_goals,half_time_away_goals,half_time_result,...,away_yellow,home_red,away_red,b365_home_win_odds,b365_draw_odds,b365_away_win_odds,market_avg_home_win_odds,market_avg_draw_odds,market_avg_away_win_odds,match_datetime
0,E0,Man United,Fulham,R Jones,H,1,0,0,0,D,...,3,0,0,1.60,4.20,5.25,1.62,4.36,5.15,2024-08-16 20:00:00
1,E0,Ipswich,Liverpool,T Robinson,A,0,2,0,0,D,...,1,0,0,8.50,5.50,1.33,8.28,5.76,1.34,2024-08-17 12:30:00
2,E0,Arsenal,Wolves,J Gillett,H,2,0,1,0,H,...,2,0,0,1.18,7.50,13.00,1.18,7.86,15.87,2024-08-17 15:00:00
3,E0,Everton,Brighton,S Hooper,A,0,3,0,1,A,...,1,1,0,2.63,3.30,2.63,2.67,3.41,2.68,2024-08-17 15:00:00
4,E0,Newcastle,Southampton,C Pawson,H,1,0,1,0,H,...,4,1,0,1.36,5.25,8.00,1.35,5.62,8.10,2024-08-17 15:00:00


In [45]:
df_market_odds.head()

,home_team,away_team,b365_home_win_odds,b365_draw_odds,b365_away_win_odds,market_avg_home_win_odds,market_avg_draw_odds,market_avg_away_win_odds
0,Man United,Fulham,1.60,4.20,5.25,1.62,4.36,5.15
1,Ipswich,Liverpool,8.50,5.50,1.33,8.28,5.76,1.34
2,Arsenal,Wolves,1.18,7.50,13.00,1.18,7.86,15.87
3,Everton,Brighton,2.63,3.30,2.63,2.67,3.41,2.68
4,Newcastle,Southampton,1.36,5.25,8.00,1.35,5.62,8.10
